# Introduction to Regular Expressions in Python

Welcome to NLP Data Preprocessing. In Machine Learning, "Garbage In, Garbage Out" is the golden rule. If your text data is full of HTML tags, punctuation, and inconsistent formatting, your NLP model will learn that noise instead of the actual linguistic signal.

### The Analytical Syntax:
RegEx relies on **Metacharacters** (characters with special mathematical meaning) and **Character Classes**:
* `\d`: Matches any digit (0-9).
* `\w`: Matches any alphanumeric character (a-z, A-Z, 0-9, _).
* `\s`: Matches any whitespace (spaces, tabs, newlines).
* `+`: Quantifier meaning "one or more" of the preceding element.
* `*`: Quantifier meaning "zero or more" of the preceding element.
* `{n,m}`: Quantifier meaning "between n and m times".

### Our Analytical Pipeline:
1. **Environment Setup:** Importing `re` and `pandas`.
2. **Core Operations:** Searching and finding patterns in raw strings.
3. **Information Extraction:** Pulling structured features (like phone numbers) from unstructured text.
4. **Data Cleaning:** Using substitution (`re.sub`) to sanitize a dataset for Machine Learning.

In [ ]:
# Cell 2: Environment Setup and Imports
import re
import pandas as pd

print("Libraries imported successfully. Ready for pattern matching.")

## Step 1: The Core Functions (`search` and `findall`)

Python's `re` module has two primary functions for data extraction:
1. `re.search(pattern, string)`: Scans the string and returns a Match object for the **first** location where the pattern produces a match.
2. `re.findall(pattern, string)`: Scans the string and returns a list of **all** non-overlapping matches.

In [ ]:
# Cell 4: Basic Searching and Finding

sample_text = "The AI model achieved 95% accuracy on dataset A, and 98% accuracy on dataset B."

# 1. Using re.search() to find the first occurrence of a pattern
# Pattern: \d+ means "one or more digits", % means "the literal percent sign"
match_object = re.search(r'\d+%', sample_text)

print("--- re.search() Results ---")
if match_object:
    print(f"Found first match: '{match_object.group()}'")
    print(f"Located at indices: {match_object.span()}")

# 2. Using re.findall() to extract all occurrences
all_matches = re.findall(r'\d+%', sample_text)

print("\n--- re.findall() Results ---")
print(f"Extracted all matches as a list: {all_matches}")

## Step 2: Information Extraction (Feature Engineering)

In ML, we often need to extract specific features from raw text to build tabular datasets. 

Let's extract structured elements like Dates and Phone Numbers using **Character Classes**. We prefix our pattern strings with `r` (e.g., `r'\d'`) to tell Python it is a "raw string", preventing Python from misinterpreting the backslashes.

In [ ]:
# Cell 6: Extracting Complex Patterns

unstructured_log = """
Customer John called on 2023-10-15 from 555-0198.
Customer Sarah emailed on 2023-11-02. Phone: 555-4021.
"""

# Pattern for YYYY-MM-DD
# \d{4} = exactly 4 digits, followed by a hyphen, etc.
date_pattern = r'\d{4}-\d{2}-\d{2}'

# Pattern for XXX-XXXX phone numbers
phone_pattern = r'\d{3}-\d{4}'

dates = re.findall(date_pattern, unstructured_log)
phones = re.findall(phone_pattern, unstructured_log)

print("--- Analytical Feature Extraction ---")
print(f"Extracted Dates:  {dates}")
print(f"Extracted Phones: {phones}")

## Step 3: Capture Groups

Sometimes a pattern is complex, but we only want to extract a specific *part* of the match. We use parentheses `()` to define a **Capture Group**.

For example, if we want to extract just the domain names from email addresses (to categorize users by company), we match the whole email but only "capture" the part after the `@` symbol.

In [ ]:
# Cell 8: Using Capture Groups

email_text = "Contact admin@corp.com or support@startup.io for help."

# Pattern breakdown:
# \w+      = one or more word characters (the username)
# @        = the literal @ symbol
# (\w+\.\w+) = CAPTURE GROUP: one or more word chars, a literal dot, and more word chars (the domain)
domain_pattern = r'\w+@(\w+\.\w+)'

# Because we used parentheses, findall() will ONLY return the contents of the group
domains = re.findall(domain_pattern, email_text)

print("--- Capture Group Results ---")
print(f"Extracted Domains for categorical feature engineering: {domains}")

## Step 4: Data Cleaning for ML (`re.sub`)

This is the most common use case for RegEx in Machine Learning. 

Before tokenizing text for a neural network, we must remove noise. `re.sub(pattern, replacement, string)` allows us to find a mathematical pattern and replace it with something else (usually an empty string `""` to delete it).

In [ ]:
# Cell 10: Cleaning a Pandas DataFrame for NLP

# Simulate a messy dataset scraped from the web
df = pd.DataFrame({
    'raw_reviews': [
        "<html>This product is GREAT!!!</html>",
        "Terrible experience... read more at http://spam.com",
        "I give it 5/5 stars! \n\n Highly recommend."
    ]
})

print("--- Original Messy Data ---")
display(df)

def clean_text(text):
    """An analytical preprocessing pipeline using RegEx."""
    
    # 1. Remove HTML tags (<anything inside>)
    text = re.sub(r'<[^>]+>', '', text)
    
    # 2. Remove URLs (http:// or https:// followed by non-whitespace)
    text = re.sub(r'https?://\S+', '', text)
    
    # 3. Remove punctuation and numbers (Keep only letters and spaces)
    # [^a-zA-Z\s] means "match anything that is NOT a letter or whitespace"
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # 4. Collapse multiple spaces/newlines into a single space
    text = re.sub(r'\s+', ' ', text)
    
    # 5. Convert to lowercase and strip leading/trailing whitespace
    return text.strip().lower()

# Apply the cleaning function to the dataframe
df['cleaned_for_ml'] = df['raw_reviews'].apply(clean_text)

print("\n--- Cleaned Data Ready for Tokenization ---")
display(df[['cleaned_for_ml']])

## Conclusion

You have successfully mastered the basics of Regular Expressions for Data Science!

**Key Analytical Takeaways:**
1. **Mathematical Text Processing:** RegEx treats text as a logical puzzle rather than a subjective sequence of characters. It is orders of magnitude faster than standard string looping.
2. **Feature Engineering:** Using `re.findall()` and capture groups allows you to extract highly specific data points (currencies, IDs, dates) from unstructured logs to build predictive tabular datasets.
3. **Sanitization:** `re.sub()` is mandatory for NLP. Feeding raw, uncleaned text into models like BERT or Word2Vec degrades performance because the model allocates parameter space to learning useless tokens like `<html>` or random URLs.